# Grid Cell Filtering Modes Demonstration

This notebook shows how different `grid_filter_mode` settings affect which grid cells are selected for extraction.

We use a real VIIRS granule over Buenos Aires and visualize the grid cells that pass each filter.

## What is AER?

AER (Asset Extraction and Retrieval) is a plugin-based geospatial data pipeline. It decouples **search** (finding satellite granules) from **extraction** (converting raw data into analysis-ready GeoTIFFs).

Core concepts you will see in every notebook:

- **`AerClient`** — The central orchestrator. It discovers plugins automatically via Python entry points, routes your search queries to the correct search provider, and delegates extraction to the correct extractor.
- **`GridDefinition`** — Creates a regular lat/lon grid (based on ESA Major TOM conventions) over any polygon. Each cell is a `GridCell` with a unique ID, UTM CRS, and a resampling footprint snapped to a target resolution grid.
- **`GridCell.area_def(resolution, padding)`** — Builds a lightweight `AreaDef` (pyresample-compatible YAML) that guarantees identical pixel alignment across different sensors when the same cell + resolution are used.
- **Grid filtering** — During `prepare_for_extraction`, AER can discard cells that are barely touched by the satellite swath, avoiding near-empty extractions. Three modes exist: `intersection`, `within`, and `coverage`.

## Setup

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from aer.client import AerClient
from aereo.grid.core import GridDefinition

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 9

## Search Results as Asset Geometry

The `search` call returns a GeoDataFrame where each row is a granule (asset) with a `geometry` column. Unlike a simple bounding box, this geometry is the **actual satellite swath footprint** (often an irregular polygon).

`GridDefinition.generate_grid_cells(aoi)` creates cells that cover the AOI. Then, during `prepare_for_extraction`, AER intersects those cells with the asset geometry — not the AOI — to decide which cells actually contain valid data. This is why a cell can be inside the AOI yet entirely outside the swath.

## Load AOI and Asset Geometry

In [ ]:
# Load Buenos Aires AOI
geojson_path = Path("../data/buenos_aires.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# Use existing VIIRS search results (or perform a new search)
client = AerClient()

results = client.search(
    collections=["VJ202IMG", "VJ203IMG"],
    start_datetime=datetime(2026, 4, 1, 15, 0, tzinfo=timezone.utc),
    end_datetime=datetime(2026, 4, 2, 15, 0, tzinfo=timezone.utc),
    intersects=aoi,
    plugin_hints={
        "VJ202IMG": "search_earthaccess",
        "VJ203IMG": "search_earthaccess",
    },
)

print(f"Found {len(results)} results")
results.head(3)

## The Three Filter Modes

AER's `prepare_for_extraction` accepts `grid_filter_mode` inside `prepare_params`:

1. **`intersection`** (default) — Keeps any cell that touches the asset geometry, even at a single point.    Maximises coverage but may include cells that are 95% NaN.
2. **`within`** — Keeps only cells fully contained inside the asset geometry.    Conservative; avoids edge effects but may drop useful partial data.
3. **`coverage`** — Keeps cells where the overlap fraction is ≥ `min_coverage` (0.0–1.0).    Balanced; tune `min_coverage` to your tolerance for missing pixels.

In this notebook we simulate the filtering logic manually so you can visualise exactly which cells survive each mode.

## Visualize Asset Geometry vs AOI

In [ ]:
# Get asset geometry for a single granule
single_asset = results.iloc[[0]]
asset_geom = single_asset.geometry.iloc[0]

fig, ax = plt.subplots(figsize=(8, 8))

# Plot AOI
gpd.GeoSeries([aoi]).plot(
    ax=ax, facecolor="none", edgecolor="black", linewidth=2, label="AOI"
)

# Plot asset geometry
gpd.GeoSeries([asset_geom]).plot(
    ax=ax, facecolor="lightblue", edgecolor="blue", alpha=0.5, label="Asset Geometry"
)

ax.set_title("VIIRS Asset Geometry vs Buenos Aires AOI")
ax.legend()
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## Coverage Heatmap Interpretation

The heatmap shows the exact overlap fraction for every cell over the AOI. Green cells (≥50%) are selected by `coverage` mode; red cells (<50%) are discarded. The percentages explain why `intersection` selects 8 cells, `within` only 3, and `coverage >= 50%` selects 5.

When you run real extraction, pass the filter like this:

```python
client.prepare_for_extraction(
    ...,
    prepare_params={
        'grid_filter_mode': 'coverage',
        'min_coverage': 0.5,
        'cells_per_task': 10,
    },
)
```

## Generate Grid Cells and Apply Filters

In [ ]:
# Generate grid cells over AOI
grid_def = GridDefinition(d=256000, overlap=False)
all_cells = list(grid_def.generate_grid_cells(aoi))

print(f"Total grid cells over AOI: {len(all_cells)}")


# Helper function to filter cells
def filter_cells(cells, asset_geom, mode, min_coverage=0.0):
    if mode == "intersection":
        return [c for c in cells if c.geom.intersects(asset_geom)]
    elif mode == "within":
        return [c for c in cells if asset_geom.contains(c.geom)]
    elif mode == "coverage":
        filtered = []
        for c in cells:
            intersection = c.geom.intersection(asset_geom)
            coverage = intersection.area / c.geom.area if c.geom.area > 0 else 0.0
            if coverage >= min_coverage:
                filtered.append((c, coverage))
        return filtered
    else:
        raise ValueError(f"Unknown mode: {mode}")


# Apply each filter
cells_intersection = filter_cells(all_cells, asset_geom, "intersection")
cells_within = filter_cells(all_cells, asset_geom, "within")
cells_coverage = filter_cells(all_cells, asset_geom, "coverage", min_coverage=0.5)

print(f"Intersection mode: {len(cells_intersection)} cells")
print(f"Within mode: {len(cells_within)} cells")
print(f"Coverage mode (>=50%): {len(cells_coverage)} cells")

## Visualize All Three Filter Modes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

modes = [
    ("intersection", cells_intersection, "Intersection\n(default)"),
    ("within", cells_within, "Within\n(fully inside)"),
    ("coverage", cells_coverage, "Coverage >= 50%\n(partial overlap)"),
]

colors = {"selected": "#2ecc71", "discarded": "#e74c3c", "asset": "#3498db"}

for ax, (mode, cells, title) in zip(axes, modes):
    # Plot AOI
    gpd.GeoSeries([aoi]).plot(
        ax=ax, facecolor="none", edgecolor="black", linewidth=2, alpha=0.3
    )

    # Plot asset geometry
    gpd.GeoSeries([asset_geom]).plot(
        ax=ax, facecolor="lightblue", edgecolor="blue", alpha=0.4, linewidth=1.5
    )

    # Plot all cells (discarded in red)
    for cell in all_cells:
        if mode == "coverage":
            cell_ids = [c[0].id() for c in cells]
        else:
            cell_ids = [c.id() for c in cells]

        if cell.id() in cell_ids:
            gpd.GeoSeries([cell.geom]).plot(
                ax=ax,
                facecolor=colors["selected"],
                edgecolor="darkgreen",
                alpha=0.6,
                linewidth=1,
            )
        else:
            gpd.GeoSeries([cell.geom]).plot(
                ax=ax,
                facecolor=colors["discarded"],
                edgecolor="darkred",
                alpha=0.3,
                linewidth=0.5,
            )

    ax.set_title(
        f"{title}\n{len(cells)} cells selected", fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")

# Add legend
legend_elements = [
    Patch(facecolor=colors["selected"], edgecolor="darkgreen", label="Selected cell"),
    Patch(facecolor=colors["discarded"], edgecolor="darkred", label="Discarded cell"),
    Patch(facecolor="lightblue", edgecolor="blue", alpha=0.4, label="Asset geometry"),
    Patch(facecolor="none", edgecolor="black", label="AOI"),
]
fig.legend(
    handles=legend_elements, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.02)
)

plt.suptitle(
    "Grid Cell Filtering Modes: VIIRS I04 over Buenos Aires",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig(
    "grid_filter_modes_comparison.png", dpi=150, bbox_inches="tight", facecolor="white"
)
plt.show()

print("Saved: grid_filter_modes_comparison.png")

## Coverage Detail View (with percentages)

In [ ]:
# Show coverage percentages for cells near the boundary
fig, ax = plt.subplots(figsize=(10, 8))

# Plot AOI and asset
gpd.GeoSeries([aoi]).plot(
    ax=ax, facecolor="none", edgecolor="black", linewidth=2, alpha=0.3
)
gpd.GeoSeries([asset_geom]).plot(
    ax=ax, facecolor="lightblue", edgecolor="blue", alpha=0.3, linewidth=1.5
)

# Plot cells with coverage color scale
all_coverage = filter_cells(all_cells, asset_geom, "coverage", min_coverage=0.0)

for cell, coverage in all_coverage:
    color = plt.cm.RdYlGn(coverage)
    gpd.GeoSeries([cell.geom]).plot(
        ax=ax, facecolor=color, edgecolor="black", alpha=0.7, linewidth=0.5
    )

    # Add text label for coverage percentage
    centroid = cell.geom.centroid
    ax.text(
        centroid.x,
        centroid.y,
        f"{coverage * 100:.0f}%",
        ha="center",
        va="center",
        fontsize=7,
        fontweight="bold",
        color="white" if coverage < 0.5 else "black",
    )

ax.set_title(
    "Cell Coverage Percentages\n(green = fully covered, red = barely covered)",
    fontsize=12,
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.6, label="Coverage Fraction")

plt.tight_layout()
plt.savefig(
    "grid_filter_coverage_detail.png", dpi=150, bbox_inches="tight", facecolor="white"
)
plt.show()

print("Saved: grid_filter_coverage_detail.png")

## Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    {
        "Filter Mode": ["intersection", "within", "coverage (>=50%)"],
        "Cells Selected": [
            len(cells_intersection),
            len(cells_within),
            len(cells_coverage),
        ],
        "Cells Discarded": [
            len(all_cells) - len(cells_intersection),
            len(all_cells) - len(cells_within),
            len(all_cells) - len(cells_coverage),
        ],
        "Description": [
            "Cell intersects asset geometry (default)",
            "Cell is fully contained within asset",
            "Cell has >=50% area inside asset",
        ],
    }
)

print(summary.to_string(index=False))